# Bài 7.2 · Phân loại âm thanh thật

**ESC-50** có 2 000 đoạn ghi âm dài 5 giây thuộc 50 loại âm thanh môi trường: chó sủa, mưa, tiếng gõ cửa, còi xe… Mỗi loại 40 đoạn, chia sẵn thành 5 fold; các đoạn cắt từ cùng một bản ghi gốc luôn nằm chung một fold, nên chấm theo fold không bị rò rỉ.

Notebook so hai cách biểu diễn một đoạn âm thanh:

1. **MFCC** tính tay, lấy trung bình và độ lệch chuẩn theo thời gian, rồi hồi quy logistic,
2. **embedding của AST** (Audio Spectrogram Transformer) đã học trên AudioSet, giữ nguyên model, chỉ học một bộ phân loại tuyến tính phía trên.

Cả hai chấm theo đúng giao thức của bộ dữ liệu: lần lượt để riêng từng fold làm test, lấy trung bình 5 lần.

Cần bật **GPU** và **Internet** trong *Session options*. Chạy hết khoảng 6 phút.

In [ ]:
import io, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import soundfile as sf, librosa, librosa.display, torch, transformers
from IPython.display import Audio, display
from transformers import AutoFeatureExtractor, ASTForAudioClassification
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('librosa', librosa.__version__, '| transformers', transformers.__version__, '| thiết bị:',
      torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')
SR = 16000

## 1. Đọc dữ liệu

Hai tệp parquet trên Hugging Face chứa cả nhãn lẫn bytes của tệp WAV gốc (44,1 kHz). Mọi đoạn được đổi về 16 kHz, đúng tần số mà AST đã học.

In [ ]:
base = 'https://huggingface.co/datasets/ashraq/esc50/resolve/main/data/'
files = ['train-00000-of-00002-2f1ab7b824ec751f.parquet', 'train-00001-of-00002-27425e5c1846b494.parquet']
t = time.time()
df = pd.concat([pd.read_parquet(base + f) for f in files], ignore_index=True)
print(f'tải xong {len(df)} đoạn sau {time.time() - t:.0f} giây')

def decode(b):
    x, sr = sf.read(io.BytesIO(b), dtype='float32')
    return librosa.resample(x, orig_sr=sr, target_sr=SR)

t = time.time()
waves = [decode(a['bytes']) for a in df['audio']]
df = df.drop(columns='audio')
print(f'giải mã và đổi về {SR} Hz sau {time.time() - t:.0f} giây | mỗi đoạn {len(waves[0]) / SR:.1f} giây')
print(df['category'].nunique(), 'loại |', df.groupby('fold').size().to_dict())
df.head()

In [ ]:
for cat in ['dog', 'rain', 'door_wood_knock']:
    i = df.index[df['category'] == cat][0]
    print(cat); display(Audio(waves[i], rate=SR))

fig, axes = plt.subplots(2, 3, figsize=(14, 5))
for j, cat in enumerate(['dog', 'rain', 'door_wood_knock']):
    i = df.index[df['category'] == cat][0]
    axes[0, j].plot(np.arange(len(waves[i])) / SR, waves[i], lw=.4, color='#3b6ea8'); axes[0, j].set_title(cat)
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=waves[i], sr=SR, n_mels=64), ref=np.max)
    librosa.display.specshow(mel, sr=SR, x_axis='time', y_axis='mel', ax=axes[1, j])
plt.tight_layout(); plt.show()

## 2. Cách 1: MFCC và hồi quy logistic

40 hệ số MFCC mỗi khung, lấy trung bình và độ lệch chuẩn theo thời gian, thành một vector 80 số cho mỗi đoạn. Cách này bỏ mất thứ tự thời gian nhưng rẻ và dễ hiểu.

In [ ]:
def mfcc_feat(x):
    m = librosa.feature.mfcc(y=x, sr=SR, n_mfcc=40)
    return np.concatenate([m.mean(1), m.std(1)])

t = time.time()
F_mfcc = np.stack([mfcc_feat(x) for x in waves])
print('MFCC:', F_mfcc.shape, f'| {time.time() - t:.0f} giây')

y = df['target'].to_numpy()

def five_fold(F):
    accs = []
    for k in sorted(df['fold'].unique()):
        tr, te = (df['fold'] != k).to_numpy(), (df['fold'] == k).to_numpy()
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5))
        clf.fit(F[tr], y[tr])
        accs.append((clf.predict(F[te]) == y[te]).mean())
    return np.array(accs)

acc_mfcc = five_fold(F_mfcc)
print('accuracy từng fold:', acc_mfcc.round(3), '→ trung bình', acc_mfcc.mean().round(3))

## 3. AST đã học trên AudioSet

AST cắt log-mel spectrogram thành các mảnh 16 × 16 rồi đưa qua Transformer, giống ViT với ảnh. Model này đã học trên AudioSet, hai triệu đoạn YouTube gán 527 nhãn, nên trước hết thử xem nó "nghe" ra gì mà chưa học thêm gì từ ESC-50.

In [ ]:
name = 'MIT/ast-finetuned-audioset-10-10-0.4593'
fe = AutoFeatureExtractor.from_pretrained(name)
ast = ASTForAudioClassification.from_pretrained(name).to(device).eval()
for cat in ['dog', 'rain', 'door_wood_knock', 'church_bells']:
    i = df.index[df['category'] == cat][0]
    inp = fe(waves[i], sampling_rate=SR, return_tensors='pt').to(device)
    with torch.no_grad():
        p = ast(**inp).logits.sigmoid()[0].cpu()
    top = p.topk(3)
    print(f'{cat:16s} →', ', '.join(f'{ast.config.id2label[int(j)]} ({v:.2f})' for v, j in zip(top.values, top.indices)))

## 4. Cách 2: embedding của AST và một bộ phân loại tuyến tính

Lấy vector 768 chiều ở cuối Transformer (`pooler_output`) cho mỗi đoạn, giữ nguyên trọng số của AST. Chỉ một hồi quy logistic được học trên các vector này, chấm theo đúng 5 fold như cách 1.

In [ ]:
t = time.time(); embs = []
with torch.no_grad(), torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
    for i in range(0, len(waves), 32):
        inp = fe(waves[i:i + 32], sampling_rate=SR, return_tensors='pt').to(device)
        embs.append(ast.audio_spectrogram_transformer(**inp).pooler_output.float().cpu())
F_ast = torch.cat(embs).numpy()
print('AST:', F_ast.shape, f'| {time.time() - t:.0f} giây')
acc_ast = five_fold(F_ast)
print('accuracy từng fold:', acc_ast.round(3), '→ trung bình', acc_ast.mean().round(3))

In [ ]:
pd.DataFrame({'cách': ['MFCC + logistic', 'AST embedding + logistic'],
              'accuracy 5 fold': [acc_mfcc.mean(), acc_ast.mean()],
              'độ lệch giữa các fold': [acc_mfcc.std(), acc_ast.std()]}).round(3)

## 5. Model còn nhầm những gì?

Gộp dự đoán của 5 lần chấm (mỗi đoạn được dự đoán đúng một lần, khi fold của nó làm test) rồi đếm các cặp hay nhầm.

In [ ]:
pred = np.zeros(len(df), dtype=int)
for k in sorted(df['fold'].unique()):
    tr, te = (df['fold'] != k).to_numpy(), (df['fold'] == k).to_numpy()
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5)).fit(F_ast[tr], y[tr])
    pred[te] = clf.predict(F_ast[te])
name_of = dict(zip(df['target'], df['category']))
wrong = y != pred
pd.Series([f'{name_of[a]} → {name_of[b]}' for a, b in zip(y[wrong], pred[wrong])]).value_counts().head(10)

## Đọc kết quả

Số dưới đây là của lần chạy mẫu trên GPU T4; chạy lại có thể lệch chút ít.

- MFCC và hồi quy logistic đạt accuracy khoảng 0,44 trên 50 lớp (đoán bừa được 0,02). MFCC tóm tắt âm sắc trung bình nên tách được những âm khác hẳn nhau, nhưng lẫn các âm có phổ gần nhau.
- Embedding của AST với cùng một bộ phân loại tuyến tính đạt khoảng 0,96, các fold dao động từ khoảng 0,94 đến 0,99. Toàn bộ khác biệt đến từ những gì AST đã học trên AudioSet; trên ESC-50 chỉ có một lớp tuyến tính được học.
- Ngay khi chưa học gì từ ESC-50, AST đã gọi đúng tên các âm: Bark và Dog cho tiếng chó, Rain cho tiếng mưa, Knock cho tiếng gõ cửa, Church bell cho tiếng chuông.
- Chấm theo 5 fold có sẵn là bắt buộc. Trộn ngẫu nhiên các đoạn sẽ đưa những mảnh cắt từ cùng một bản ghi vào cả train lẫn test, và điểm cao giả.

Gợi ý khi demo: ghi âm một tiếng động trong lớp (vỗ tay, gõ bàn), tải lên Kaggle rồi cho AST ở mục 3 nghe thử.